# Dreamer World Model Homework: Public Colab Template

This notebook clones your GitHub copy of the homework, installs dependencies, trains `MiniDreamer`, evaluates the policy, and scores the public world-model benchmark.


## 1. Configure Your Repository

Fork or copy the course repository first. Then replace `COURSE_REPO_URL` with your own repository so Colab edits are not lost when the runtime disconnects.


In [ ]:
from pathlib import Path
import subprocess
import sys

COURSE_REPO_URL = "https://github.com/WeijieLai1024/EEC289A_WorldModel-Homework.git"
COURSE_REPO_BRANCH = "main"
COURSE_REPO_DIR = Path("/content/world_model_course_repo")

def run(cmd, cwd=None):
    cmd = [str(part) for part in cmd]
    print("+", " ".join(cmd))
    return subprocess.run(cmd, cwd=cwd, check=True)

def ensure_course_repo(repo_url, branch, target_dir):
    if target_dir.exists():
        print(f"Using existing repo at {target_dir}")
        return
    run(["git", "clone", "--branch", branch, repo_url, target_dir])

print("Repo target:", COURSE_REPO_DIR)


## 2. Install Dependencies And Clone Repo


In [ ]:
!python -m pip install -q -U pip setuptools wheel
ensure_course_repo(COURSE_REPO_URL, COURSE_REPO_BRANCH, COURSE_REPO_DIR)
%cd {COURSE_REPO_DIR}
!python -m pip install -q -r configs/colab_requirements.txt

import torch, gymnasium
print("torch:", torch.__version__)
print("gymnasium:", gymnasium.__version__)
print("cuda available:", torch.cuda.is_available())


## 3. Inspect The Environment


In [ ]:
%cd {COURSE_REPO_DIR}
!python inspect_env.py


## 4. Read The Important Files

The assignment is designed so the core implementation is readable. Start here before changing hyperparameters.


In [ ]:
!sed -n '1,240p' world_model_hw/models.py


In [ ]:
!sed -n '1,260p' world_model_hw/agent.py


## 5. Create A Colab Runtime Config

Use `DEBUG_RUN = True` for a quick sanity check. Use `False` for the baseline budget.


In [ ]:
import json

DEBUG_RUN = True
base_config_path = COURSE_REPO_DIR / "configs" / "course_config.json"
config = json.loads(base_config_path.read_text())

if DEBUG_RUN:
    stage = "local_smoke"
else:
    stage = "baseline"

config["colab_notes"] = {
    "debug_run": DEBUG_RUN,
    "stage": stage,
    "runtime": "Colab"
}
runtime_config_path = COURSE_REPO_DIR / "configs" / "colab_runtime_config.json"
runtime_config_path.write_text(json.dumps(config, indent=2))
print("wrote", runtime_config_path)
print("stage", stage)


## 6. Dry Run


In [ ]:
!python train.py --config configs/colab_runtime_config.json --stage {stage} --dry-run


## 7. Train


In [ ]:
output_dir = "artifacts/smoke" if DEBUG_RUN else "artifacts/run_baseline"
!python train.py --config configs/colab_runtime_config.json --stage {stage} --output-dir {output_dir}


## 8. Evaluate Policy And Render Demo


In [ ]:
checkpoint_dir = COURSE_REPO_DIR / output_dir / "best_checkpoint"
!python evaluate_policy.py --checkpoint-dir {checkpoint_dir} --render --output-dir artifacts/demo_bundle


## 9. World Model Prediction Sanity Check


In [ ]:
!python quick_world_model_check.py --checkpoint-dir {checkpoint_dir} --output-dir artifacts/demo_bundle


## 10. Public Benchmark


In [ ]:
!python generate_public_rollout.py --checkpoint-dir {checkpoint_dir} --output-dir artifacts/public_eval_bundle --num-episodes 8 --render-first-episode
!python public_eval.py --config configs/colab_runtime_config.json --rollout-npz artifacts/public_eval_bundle/rollout_public_eval.npz --output-json artifacts/public_eval_bundle/public_eval.json


## 11. Save Work Back To GitHub

Run these commands after editing files in Colab:

```bash
git status
git add .
git commit -m "Improve MiniDreamer homework solution"
git push origin main
```
